# Neural Radiance Fields (NeRF) & Fourier Positional Encoding

## 1. Representing 3D Scenes as Continuous Neural Fields
Neural Radiance Fields (Mildenhall et al., ECCV 2020) represent a 3D scene not as discrete voxels, point clouds, or triangle meshes, but as a **continuous function parameterized by an MLP**:

$$F_\Theta: (x, y, z, \theta, \phi) \mapsto (r, g, b, \sigma)$$

Where:
- $\mathbf{x} = (x, y, z)$ is the 3D spatial position
- $\mathbf{d} = (\theta, \phi)$ is the viewing direction unit vector
- $(r, g, b)$ is emitted radiance color
- $\sigma$ is volume differential density

### The Spectral Bias of MLPs & Fourier Positional Encoding
Standard coordinate-based MLPs $F_\Theta(\mathbf{x})$ suffer from **spectral bias**: Neural Tangent Kernel (NTK) theory proves that deep networks prioritize low-frequency variations, rendering them incapable of synthesizing sharp geometric edges, fine shadows, and high-frequency textures.

To resolve this, NeRF maps low-dimensional coordinates into a higher-dimensional Fourier basis:

$$\gamma(p) = \left( \sin(2^0 \pi p), \cos(2^0 \pi p), \sin(2^1 \pi p), \cos(2^1 \pi p), \dots, \sin(2^{L-1} \pi p), \cos(2^{L-1} \pi p) \right)$$

For $p \in \mathbb{R}^3$ with $L=10$ frequency bands, coordinates are projected into $\mathbb{R}^{3 \times 20} = \mathbb{R}^{60}$, enabling the MLP to fit extreme high-frequency details!

In [ ]:
import numpy as np

def fourier_positional_encoding(coords, num_frequencies=10):
    """
    Maps 3D spatial coordinates [N, 3] to Fourier features [N, 3 * 2 * num_frequencies].
    """
    freq_bands = 2.0 ** np.arange(num_frequencies) # [2^0, 2^1, ..., 2^(L-1)]
    encoded = []
    for freq in freq_bands:
        for fn in [np.sin, np.cos]:
            encoded.append(fn(coords * np.pi * freq))
    return np.concatenate(encoded, axis=-1)

def volumetric_rendering(colors, densities, deltas):
    """
    Discretized Volumetric Rendering Equation:
    C(r) = sum_{i=1}^N T_i * (1 - exp(-sigma_i * delta_i)) * c_i
    Where T_i = exp(-sum_{j=1}^{i-1} sigma_j * delta_j)
    """
    # colors: [N_samples, 3]
    # densities: [N_samples]
    # deltas: [N_samples] (step distance between consecutive ray samples)
    
    alpha = 1.0 - np.exp(-densities * deltas) # [N_samples]
    
    # Transmittance T_i: probability that ray travels up to sample i without hitting anything
    accum_density = np.cumsum(densities * deltas)
    T = np.exp(-np.pad(accum_density[:-1], (1, 0), constant_values=0.0))
    
    weights = T * alpha # [N_samples]
    accumulated_color = np.sum(weights[:, None] * colors, axis=0)
    return accumulated_color, weights

# Verification
coords = np.array([[0.1, -0.5, 0.8], [0.0, 0.2, -0.4]])
encoded_coords = fourier_positional_encoding(coords, num_frequencies=6)
print(f"Original 3D Coordinates Shape : {coords.shape}")
print(f"Fourier Encoded Vector Shape  : {encoded_coords.shape} (Expanded from 3 to {coords.shape[1] * 2 * 6} dims)")

# Test Volumetric Ray Rendering
num_samples = 64
sample_colors = np.ones((num_samples, 3)) * 0.8
# High density object located in middle samples (idx 30 to 35)
sample_densities = np.zeros(num_samples)
sample_densities[30:35] = 5.0
deltas = np.ones(num_samples) * 0.05

ray_rgb, sample_weights = volumetric_rendering(sample_colors, sample_densities, deltas)
print(f"Accumulated Ray RGB Color     : {np.round(ray_rgb, 3)}")
print(f"Total Opacity / Weight Sum    : {np.sum(sample_weights):.3f} (Ray fully absorbed by opaque object)")

## 2. NeRF Rendering Pipeline Architecture
$$\text{Ray } \mathbf{r}(t) = \mathbf{o} + t\mathbf{d} \xrightarrow{\text{Sample } t_i} \mathbf{x}_i \xrightarrow{\gamma(\mathbf{x}_i)} \text{MLP} \to (\sigma_i, \mathbf{c}_i) \xrightarrow{\text{Volumetric Numerical Integral}} C(\mathbf{r})$$

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why does 3D Gaussian Splatting (3DGS) train and render significantly faster than NeRF?
**Answer:** NeRF queries a multi-layer perceptron hundreds of times along every single ray for every pixel, requiring trillions of MLP forward passes per second. **3D Gaussian Splatting (SIGGRAPH 2023)** completely eliminates neural network evaluation at inference: it represents scenes with millions of explicit 3D Gaussians (position, covariance, opacity, spherical harmonics) and projects them directly to screen space using a hardware-accelerated tile-based rasterizer, achieving real-time rendering at $>100$ FPS at 1080p resolution.

### Q2: Why is viewing direction $\mathbf{d}$ fed only into the final layers of the NeRF MLP?
**Answer:** Differential volume density $\sigma$ is an intrinsic physical property of the geometry and must be **view-independent** (an opaque wall is solid regardless of angle). Radiance color $(r, g, b)$, however, includes specular highlights and reflections that are strictly **view-dependent**. Passing $\mathbf{d}$ only to the final color prediction layer enforces that geometry is invariant to camera perspective.